# *The Cannon*: Getting Started (Demo with APOGEE Data)

[*The Cannon*](https://arxiv.org/abs/1501.07604) (Ness et al. 2015) is a data-driven
model for stellar spectra. It assumes that a star's spectrum is a smooth function of (quadratic functions of) stellar labels like effective temperature, surface gravity, and element abundances. In typical usage, one defines a training set of reference stars for which both the spectrum and the labels are already known, and then the trained model can be run backwards to infer the labels for new spectra. Unlike _Lux_ and other latent variable models, it assumes that the labels are perfectly known for the training stars and can't handle missing labels in the training set. However, it is very fast and effective for many "label transfer" problems.

In Pollux, we can implement _the Cannon_ as a particular architecture of a [latent variable model](LVM-math-notes.ipynb). However, we provide a pre-defined implementation in {py:class}`~pollux.models.Cannon`.

What makes _the Cannon_ distinct from other LVMs we can implement in Pollux is that the latent space is not free: the latents _are_ the quadratic features built from the stellar labels. In the [*Lux* tutorial](Lux-getting-started-apogee.ipynb), which fits the same APOGEE data with a _Lux_ model instead, the latent vectors are nuisance parameters that map to both the labels and the spectra, which means we can handle uncertainties on the labels and missing labels in the training set. However, _Lux_ is a bilinear model, whereas _the Cannon_ is quadratic in the labels, so it can capture more complex relationships between the labels and the spectra. 

We fit the same 1000 high signal-to-noise APOGEE (DR17) red giants (which are truncated to contain data for one of three detector chips) and do the same train/test split as in the *Lux* tutorial, so we can compare the prediction accuracies at the end.

In [ ]:
import itertools

import astropy.table as at
import h5py
import jax
import matplotlib.pyplot as plt
import numpy as np
import numpyro
from astropy.stats import median_absolute_deviation as MAD

import pollux as plx

jax.config.update("jax_enable_x64", True)
%matplotlib inline

## Load the data

We have pre-processed the data and stored it in an HDF5 file ([available here](https://users.flatironinstitute.org/~apricewhelan/pollux/rgb-highSNR-1k-1chip.h5)) along with the corresponding rows from the APOGEE "allStar" catalog file. 

In [ ]:
with h5py.File("../_data/rgb-highSNR-1k-1chip.h5", "r") as f:
    apid = f["APSTAR_ID"][:]
    allstar = at.Table.read(f, path="allStar")
    assert np.all(allstar["APSTAR_ID"] == apid)

    all_wvln = f["wavelength"][:].astype("f8")
    all_flux = f["flux"][:].astype("f8")
    all_flux_err = f["flux_err"][:].astype("f8")

We now do some quality cuts on the spectral data. The flux data is a 2D array - one dimension corresponds to "stars" and the other "pixels" (wavelength):

In [ ]:
all_flux.shape

These are only minimal quality cuts because the stars have been pre-selected to have high signal-to-noise, so if you are working with a more heterogeneous APOGEE or other stellar spectral data set, you might need to do a more careful filtering. Here we require that the flux and flux error values are finite and that the error is positive. We also remove any pixels from the wavelength grid where >75% of stars have low SNR (which usually indicates bad pixels).

In [ ]:
pix_snr = all_flux / all_flux_err

pixel_remove_mask = (
    # remove pixels that are the same for all stars (probably bad values)
    np.all(all_flux == all_flux[0], axis=0)
    # remove pixels where >75% of stars have low SNR (probably bad pixels)
    | ((pix_snr < 5).sum(axis=0) > 0.75 * pix_snr.shape[0])
)
flux = all_flux[:, ~pixel_remove_mask]
flux_err = all_flux_err[:, ~pixel_remove_mask]
wvln = all_wvln[~pixel_remove_mask]

# replace locations with zero or negative flux errors, or bad flux values:
bad_flux_mask = (all_flux_err <= 0) | (~np.isfinite(all_flux))
flux[bad_flux_mask] = 1.0
flux_err[bad_flux_mask] = 1e10  # set to large error to effectively ignore these pixels

Let's now visualize a few random spectra after this filtering:

In [ ]:
rng = np.random.default_rng(123)
rand_idx = rng.choice(flux.shape[0], size=5, replace=False)

fig, ax = plt.subplots(figsize=(12, 4))
for i in rand_idx:
    _ = ax.plot(wvln, flux[i], marker="", drawstyle="steps-mid", lw=0.75)

_ = ax.set(xlabel=r"wavelength [$\AA$]", ylabel="normalized flux", ylim=(0.5, 1.2))

And here is a view of the label data (stellar parameters and abundances) for these stars:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5), layout="constrained")

axes[0].hist2d(
    allstar["TEFF"],
    allstar["LOGG"],
    bins=(np.linspace(3000, 6000, 64), np.linspace(0.5, 4.0, 64)),
    cmap="magma_r",
)
axes[0].set(xlim=(6000, 3000), ylim=(4.0, 0.5), xlabel="TEFF", ylabel="LOGG")

axes[1].hist2d(
    allstar["FE_H"],
    allstar["MG_FE"],
    bins=(np.linspace(-2.0, 0.5, 64), np.linspace(-0.2, 0.6, 64)),
    cmap="magma_r",
)
_ = axes[1].set(xlim=(-2.0, 0.5), ylim=(-0.2, 0.6), xlabel="FE_H", ylabel="MG_FE")

## Assemble the data

The flux data is already a 2D array of stars against pixels; we need the labels in the
same shape, one row per star. We will use four labels: effective temperature, surface
gravity, overall metallicity, and the $\alpha$-element enhancement.

In [ ]:
label_names = ["TEFF", "LOGG", "M_H", "ALPHA_M"]
labels = np.array([allstar[name] for name in label_names]).T.astype("f8")
label_errs = np.array([allstar[f"{name}_ERR"] for name in label_names]).T.astype("f8")

### The labels need to be rescaled

This is the one place where _the Cannon_'s data handling differs from that in _Lux_.

The labels enter the model through a polynomial expansion. Teff is a number near
4600, while `ALPHA_M` is a number near 0. Squared and cross-multiplied, those become large values. The normal equations for the coefficients then involve a matrix that could become numerically singular. So here we attach a {py:class}`~pollux.data.ShiftScalePreprocessor` to the labels, which centers and scales each one before it is expanded into features. 

The flux is the opposite case. Each pixel's flux is generated by its own row of $\theta$, so shifting and scaling a pixel is absorbed exactly into that row and the fitted jitter. Since APOGEE spectra are already continuum-normalized near 1.0, we leave them in their native units, as the [*Lux* tutorial](Lux-getting-started-apogee.ipynb) does, so that the fitted scatter is readable as a fraction of the continuum.

In [ ]:
all_data = plx.data.PolluxData(
    # native units: a per-pixel shift and scale would be absorbed into theta anyway
    flux=plx.data.OutputData(flux, err=flux_err),
    # rescaled: these get raised to powers, so their scale matters
    label=plx.data.OutputData(
        labels,
        err=label_errs,
        preprocessor=plx.data.ShiftScalePreprocessor.from_data(labels),
    ),
)

preprocessed_data = all_data.preprocess()
n_stars = len(all_data)
n_labels = len(label_names)
n_flux = len(wvln)
print(f"{n_stars=}, {n_labels=}, {n_flux=}")

We now split the data into training and test sets using random indices:

In [ ]:
idx = np.arange(n_stars)
rng.shuffle(idx)

train_idx = idx[: 3 * n_stars // 4]
test_idx = idx[3 * n_stars // 4 :]

# `_unproc` holds the labels in their catalog units, for scoring predictions later
train_data_unproc = all_data[train_idx]
test_data_unproc = all_data[test_idx]

train_data = preprocessed_data[train_idx]
test_data = preprocessed_data[test_idx]
len(train_data), len(test_data)

We now pretend that we are missing label information for the test data, so that
inferring it back is a fair test:

In [ ]:
test_flux_only = plx.data.PolluxData(flux=test_data["flux"])

## Set up the model

{py:class}`~pollux.models.Cannon` assembles the architecture described in the introduction. Two outputs are registered:

- `label`: observed directly through a {py:class}`~pollux.models.transforms.NoOpTransform`. This is the piece that makes the latents *be* the labels: the model asserts that the catalog labels are noisy observations of the latent vector itself, with no map in between.
- `flux`: generated by a {py:class}`~pollux.models.transforms.PolyFeatureTransform` (which expands the labels into the $g_j$) followed by a {py:class}`~pollux.models.transforms.LinearTransform` (which applies $\theta$).

As we did in the [*Lux* tutorial](Lux-getting-started-apogee.ipynb) and the [error models tutorial](LVM-error-models.ipynb), we also fit a per-pixel intrinsic scatter for the flux. This is added in quadrature to the reported flux errors, and it absorbs both globally underestimated uncertainties and the model's own inability to reproduce particular pixels. We set the prior scale directly in flux units, since we did not rescale the flux. The catalog label errors are taken at face value (but we try with and without handling these uncertainties).

In [ ]:
model = plx.Cannon(
    label_size=n_labels,
    output_size=n_flux,
    poly_degree=2,
    # a per-pixel jitter for the flux, with a HalfNormal(0.1) prior in flux units
    intrinsic_scatter={"flux": 0.1},
)
print(
    f"{model.n_features} polynomial features, so theta has shape "
    f"({n_flux}, {model.n_features})"
)

## Train the model

For the training stars, the labels are data, not unknowns. In the traditional usage of _the Cannon_, the labels are held fixed at their catalog values (i.e. label uncertainties are ignored) for the training set while we infer the coefficients $\theta$. So the training step below is the classic Cannon step: we hold the labels at their catalog values and solve for the coefficients (using `fixed_pars`). Because the flux is linear in $\theta$, that solve is a weighted least squares problem with an exact solution, which {py:func}`~pollux.models.optimize_iterative` will recognize and use. However, we also infer the intrinsic scatter (the jitter), so that must be optimized using gradient descent --- the fit alternates between the two blocks 

In [ ]:
# The labels are observations, so they are inputs to the training fit, not parameters:
observed_labels = np.asarray(train_data["label"].data)

results = model.optimize_iterative(
    train_data,
    # theta: one exact weighted-least-squares solve. s: gradient descent.
    blocks=["flux:data", "flux:err"],
    fixed_pars={"latents": observed_labels},
    max_cycles=8,
    rng_key=jax.random.PRNGKey(123),
    block_options={
        "flux:err": {
            "optimizer": numpyro.optim.Adam,
            "optimizer_kwargs": {"step_size": 1e-2},
            # note: num_steps is a block setting, not an argument to the optimizer
            "num_steps": 1000,
        }
    },
    progress=False,
)
opt_pars = results.params
print(f"converged after {len(results.losses_per_cycle)} cycles")

### What happens if you let the training labels float

While the traditional _Cannon_ approach is to hold the training labels fixed, Pollux allows you to handle uncertainties in the training labels. Here we demonstrate how to do that. We demonstrate by starting from the good fit values above, and then running the training again with the training labels as free parameters. 

In [ ]:
floated = model.optimize_iterative(
    train_data,
    initial_params=opt_pars,  # start from the good fit above
    max_cycles=3,  # this is already enough to see the problem
    rng_key=jax.random.PRNGKey(123),
    block_options={
        "flux:err": {
            "optimizer": numpyro.optim.Adam,
            "optimizer_kwargs": {"step_size": 1e-2},
            "num_steps": 1000,
        }
    },
    progress=False,
)
floated_opt_pars = floated.params
print(f"converged after {len(floated.losses_per_cycle)} cycles")

## Inferring labels for new spectra

With $\theta$ and the jitter fixed, we can now infer the labels for the test set (which we are pretending that we only have spectra for). 

Unlike in *Lux*, this is not a linear solve. The spectrum is linear in $\theta$, but it is a polynomial in the labels, and the labels are what we are solving for now. So there is no closed form and we use gradient descent. {py:func}`~pollux.models.optimize_iterative` will emit a {py:class}`~pollux.exceptions.PolluxLinearizationWarning` saying that `flux` "is not affine in the latents" and falls back to Adam. That warning is expected for a _Cannon_ model with `poly_degree` above 1, and it is about the `latents` block specifically. Here we use {py:func}`~pollux.models.optimize` instead.

In [ ]:
# Everything except the per-object latents: theta and the flux jitter
fixed_pars = model.output_pars(opt_pars)

test_opt_pars, test_svi = model.optimize(
    test_flux_only,
    fixed_pars=fixed_pars,
    names=["flux"],  # there are no labels to fit against, only spectra
    num_steps=4_000,
    optimizer=numpyro.optim.Adam(1e-2),
    rng_key=jax.random.PRNGKey(42),
    svi_run_kwargs={"progress_bar": False},
)
test_svi.losses.block_until_ready()  # make sure it runs now

The inferred latents are the predicted labels, in rescaled units, so we undo the label preprocessing to get physical units:

In [ ]:
predict_test_values = model.predict_outputs(
    fixed_pars, latents=test_opt_pars["latents"]
)
predict_test_values = test_data.unprocess(predict_test_values)

Now we compare the _Cannon_-predicted labels for the test set against the "true" (i.e.
APOGEE catalog) values:

In [ ]:
pt_style = {"ls": "none", "ms": 2.0, "alpha": 0.5, "marker": "o", "color": "k"}

fig, axes = plt.subplots(1, n_labels, figsize=(4 * n_labels, 4.5), layout="constrained")
for i in range(n_labels):
    axes[i].errorbar(
        predict_test_values["label"].data[:, i],
        test_data_unproc["label"].data[:, i],
        yerr=test_data_unproc["label"].err[:, i],
        **pt_style,
    )
    axes[i].set(xlabel=f"Cannon {label_names[i]}", ylabel=f"ASPCAP {label_names[i]}")
    _val = np.median(test_data_unproc["label"].data[:, i])
    axes[i].axline([_val, _val], slope=1, color="tab:green", zorder=-100, alpha=0.5)

_ = fig.suptitle("Test set: Cannon-predicted vs. APOGEE labels", fontsize=22)

The predictions track the catalog values well, with a few outliers. Note that the non-convexity we discussed above (the polynomial solve) has a second consequence here: label inference has more than one local optimum. Most stars land in the right one from a cold start, a few will not. Passing `initial_params` with a rough first guess at the labels usually fixes these failures.

For a quantitative measure, we compute the root-mean-square error and a robust scatter per label, and --- so that the labels can be compared to one another and to the [*Lux* results](Lux-getting-started-apogee.ipynb) --- the RMSE in units of each label's median catalog uncertainty:

In [ ]:
resid = np.asarray(predict_test_values["label"].data) - np.asarray(
    test_data_unproc["label"].data
)
err = np.asarray(test_data_unproc["label"].err)

rmse = np.sqrt(np.mean(resid**2, axis=0))
mad_std = 1.5 * MAD(resid, axis=0)
median_err = np.median(err, axis=0)

for i in range(n_labels):
    print(
        f"{label_names[i]:>8s}: RMSE = {rmse[i]:7.3f}, robust = {mad_std[i]:7.3f}, "
        f"median err = {median_err[i]:6.3f}, RMSE/err = {rmse[i] / median_err[i]:5.2f}"
    )
print(f"\nmean RMSE / catalog error = {np.mean(rmse / median_err):.2f}")

As in the *Lux* tutorial, the predictions scatter several times more than the catalog
uncertainties alone would imply. Two caveats on reading that ratio: the catalog
uncertainties are themselves likely underestimated, and they come from fitting the entire
APOGEE spectrum rather than the single detector chip we are using here. The ratio is
still the right diagnostic, because it is scale-free and can be computed for every label
--- and it is what we will use below to choose the polynomial degree.

## What the model learned

The _Cannon_ has one advantage over a free-latent model like *Lux*, which is that the parameters (i.e. the coefficients $\theta$) have a clear physical interpretation. Each column of $\theta$ is a spectrum-like map of how sensitive every pixel is to one particular label. The column multiplying the constant feature is the mean spectrum, the column multiplying Teff shows which lines respond most to temperature, and so on.

*Lux* alone does not have the same interpretability in the raw parameters. The latent space is only defined up to a rotation, so an individual column of the $A$ matrix has no standalone interpretation unless we impose additional constraints (like non-negativity). So, _The Cannon_ is less flexible and slightly harder to optimize, but it is more inherently interpretable.

In [ ]:
# theta lives in the linear layer, which is the second transform in the flux sequence
theta = np.asarray(opt_pars["flux"]["data"][1]["A"])

# feature names, in the order PolyFeatureTransform generates them
feature_names = ["1"]
for deg in range(1, model.poly_degree + 1):
    feature_names += [
        " ".join(c) for c in itertools.combinations_with_replacement(label_names, deg)
    ]

fig, axes = plt.subplots(
    n_labels + 1, 1, figsize=(12, 10), sharex=True, layout="constrained"
)
# the constant term plus the four first-order terms
for ax, j in zip(axes, range(n_labels + 1)):
    ax.plot(wvln, theta[:, j], marker="", drawstyle="steps-mid", lw=0.75, color="k")
    ax.set_ylabel(feature_names[j], fontsize=11)
    ax.axhline(0, color="tab:red", lw=0.5, zorder=-10)

axes[0].set_title(r"Cannon coefficient spectra $\theta$: constant and linear terms")
_ = axes[-1].set(xlabel=r"wavelength [$\AA$]")

The top panel is the mean spectrum. Below it, each panel is a derivative: where it
departs from zero, that pixel carries information about that label. The temperature panel
responds broadly across the chip, while the abundance panels spike at a handful of
specific lines --- which is exactly the behavior that makes label inference from spectra
possible at all.

The fitted jitter is the complementary diagnostic. It tells us where the polynomial model
*failed*, and since we left the flux in native units it reads directly as a fraction of
the continuum:

In [ ]:
flux_s = np.asarray(opt_pars["flux"]["err"]["s"])

fig, ax = plt.subplots(figsize=(12, 4), layout="constrained")
ax.plot(wvln, flux_s, marker="", drawstyle="steps-mid", lw=0.75, color="#555555")
_ = ax.set(
    xlabel=r"wavelength [$\AA$]",
    ylabel="fitted jitter $s$",
    title="Where the Cannon model cannot reproduce the data",
)
print(f"jitter: median = {np.median(flux_s):.4f}, max = {flux_s.max():.4f}")
print(f"smallest fitted jitter: {flux_s.min():.5f} (zero would mean a collapsed pixel)")

The median jitter is a fraction of a percent of the continuum, with excursions at specific pixels where a degree-2 polynomial in four labels cannot capture the real variation --- lines driven by an abundance we did not include, for example.

One note for context from the [error models tutorial](LVM-error-models.ipynb): fitting a variance has a degenerate optimum at zero, and exact alternating solves can fall into it, which is why the *Lux* tutorial avoids {py:meth}`~pollux.models.LVM.optimize_iterative` on this data. That risk is absent here because the jitter block never gets a closed-form solve.

## Choosing the polynomial degree

`poly_degree` is the Cannon's capacity knob, and it plays the same role that
`n_latents` plays in *Lux*: too low and the model cannot represent the real
spectrum-label relationship, too high and it fits noise in the training set. The
[latent dimensionality section](Lux-getting-started-apogee.ipynb) of the *Lux* tutorial
makes the same argument for its own hyperparameter, and the method is identical --- refit
at several values, score each on held-out data.

Note how much narrower the range is. Degree $d$ with 4 labels gives
$\binom{4+d}{d}$ features, so the count climbs steeply: 5, 15, 35, 70, 126, 210. *Lux*
could afford 96 latent dimensions; the Cannon runs out of training stars much sooner.

In [ ]:
poly_degrees = [1, 2, 3, 4, 5]

all_rmse = []
for degree in poly_degrees:
    model_i = plx.Cannon(
        label_size=n_labels,
        output_size=n_flux,
        poly_degree=degree,
        intrinsic_scatter={"flux": 0.02},
    )
    res_i = model_i.optimize_iterative(
        train_data,
        blocks=["flux:data", "flux:err"],
        fixed_pars={"latents": observed_labels},
        max_cycles=8,
        rng_key=jax.random.PRNGKey(123),
        block_options={
            "flux:err": {
                "optimizer": numpyro.optim.Adam,
                "optimizer_kwargs": {"step_size": 1e-2},
                "num_steps": 500,
            }
        },
        progress=False,
    )

    fixed_i = model_i.output_pars(res_i.params)
    test_pars_i, _ = model_i.optimize(
        test_flux_only,
        fixed_pars=fixed_i,
        names=["flux"],
        num_steps=4_000,
        optimizer=numpyro.optim.Adam(1e-2),
        rng_key=jax.random.PRNGKey(42),
        svi_run_kwargs={"progress_bar": False},
    )
    pred_i = test_data.unprocess(
        model_i.predict_outputs(fixed_i, latents=test_pars_i["latents"])
    )
    resid_i = np.asarray(pred_i["label"].data) - np.asarray(
        test_data_unproc["label"].data
    )
    all_rmse.append(np.sqrt(np.mean(resid_i**2, axis=0)))
    print(
        f"degree {degree}: {model_i.n_features:>3d} features, "
        f"mean RMSE/err = {np.mean(all_rmse[-1] / median_err):5.2f}"
    )

all_rmse = np.array(all_rmse)  # (n_degrees, n_labels), in physical units
all_rmse_scaled = all_rmse / median_err

In [ ]:
mean_rmse = np.mean(all_rmse_scaled, axis=1)
best_degree = poly_degrees[np.argmin(mean_rmse)]
print(f"best poly_degree = {best_degree}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), layout="constrained")

for i, name in enumerate(label_names):
    axes[0].plot(poly_degrees, all_rmse_scaled[:, i], marker="o", label=name)
axes[0].set(
    xlabel="poly_degree",
    ylabel="RMSE / median catalog error",
    title="Per-label test RMSE",
    yscale="log",
)
axes[0].legend(fontsize=9)

axes[1].plot(poly_degrees, mean_rmse, marker="o", color="k")
axes[1].set(
    xlabel="poly_degree",
    ylabel="mean RMSE / catalog error",
    title="Held-out validation summary",
)

for ax in axes:
    ax.axvline(best_degree, color="#aaaaaa", ls="--", lw=1.5, zorder=-10)

_ = fig.suptitle("Choosing poly_degree on held-out data", fontsize=20)

The curve has the shape the argument predicts. Degree 1 is badly underfit (a purely linear model of the spectrum misses most of the signal) and the improvement from 2 to 4 is modest. Beyond that, the feature count grows faster than the training set can support and accuracy degrades.

The same caveat as in the *Lux* tutorial applies: this is validation on a single held-out split, not k-fold cross-validation, so where the curve flattens is a more trustworthy readout than exactly which point is lowest.

### Where to go next

- [*Lux*: Getting Started](Lux-getting-started-apogee.ipynb) --- the free-latent model on this same data, including the affine-layer trick for skipping preprocessing.
- [Latent variable models](LVM-math-notes.ipynb) --- the objective function and the bilinear structure both models are built on.
- [LVM: Getting Started](LVM-getting-started.ipynb) --- the same ideas on simulated data, where the truth is known, plus a fuller treatment of {py:func}`~pollux.models.optimize_iterative`.
- [Error models](LVM-error-models.ipynb) --- fitting an unknown intrinsic scatter, and the variance-collapse failure mode to watch for.